In [1]:
import pandas as pd
import numpy as np

## Load cleaned poker decisions
## Relative path allows the notebook to run on Windows or Linux
filepath = "parquet/decisions/handhq/ABS-2009-07-01_2009-07-23_1000NLH_OBFU/10/part-0000.parquet"
df = pd.read_parquet(
    filepath,
    columns=["hand_id", "street", "to_call", "action"]
)

## Identify whether the player is facing a bet
df["facing_bet"] = df["to_call"] >0

## Display the first five observations
print(df.head())


   hand_id  street  to_call  action  facing_bet
0        0       0     10.0       2        True
1        0       0     47.5       0        True
2        0       0     47.5       0        True
3        0       0     47.5       0        True
4        0       0     42.5       0        True


In [2]:
## Separate poker hands into training and testing sets
hand_ids = df["hand_id"].unique()

##shuffle the hand IDs for a random split
np.random.seed(42)## fix random seed so training/testing splits are reproducible
np.random.shuffle(hand_ids)

## Use 80% for training and 20% for testing
cutoff = int(len(hand_ids)*0.8)

train = df[df["hand_id"].isin(hand_ids[:cutoff])]##contains decisions to estimate action probabilities
test = df[df["hand_id"].isin(hand_ids[cutoff:])]##contains decisions to measure accuracy

##check the number of decisions in each set
print("Training decisions:", len(train))
print("Testing decisions:", len(test))

Training decisions: 334007
Testing decisions: 83145


In [3]:
##Poker stage labels
street_labels ={
    0: "Pre-flop",
    1: "Flop",
    2: "Turn",
    3: "River"
}

##Poker action labels
action_labels = {
    0: "Fold",
    1: "Check/call",
    2: "Bet/Raise"
}


In [4]:
##Counts how often each action occurs for each poker situation
counts = pd.crosstab([train["street"], train["facing_bet"]],train["action"])

##Include all three possible actions
counts = counts.reindex(columns=[0,1,2], fill_value=0)

##Bayesian update: apply Dirichlet prior (1,1,1)
alpha = counts + 1

##Calculate posterior mean probabilities
##alpha.sum(axis=1)calculates the total updated count for each situation
##Dividing each count by its row total gives the estimated probability of each action
probabilities = alpha.div(alpha.sum(axis=1),axis=0)

##Replace numeric labels in the displayed table
labeled_probabilities = probabilities.rename(
    index=street_labels,
    level="street"
)

labeled_probabilities = labeled_probabilities.rename(
    columns=action_labels
)
print (labeled_probabilities)

action                   Fold  Check/call  Bet/Raise
street   facing_bet                                 
Pre-flop False       0.002740    0.760031   0.237228
         True        0.656108    0.137500   0.206393
Flop     False       0.000404    0.609603   0.389992
         True        0.524156    0.348256   0.127588
Turn     False       0.000091    0.631047   0.368862
         True        0.493611    0.405570   0.100819
River    False       0.000428    0.638550   0.361022
         True        0.546413    0.367156   0.086430


In [5]:
## Select the most likely action for each poker situation
##idxmax is a pandas method that finds the label associated with the largest value
most_likely = probabilities.idxmax(axis=1).reset_index(name="prediction")

## Match predictions to the testing data
##pandas method that combines 2 tables with matching cols
results = test.merge(most_likely, on=["street", "facing_bet"], how="left")

##Calculate prediction accuracy
accuracy = (results["prediction"]==results["action"]).mean()

##Baseline: always predict the most common training action
##train["action"] — Selects the actual poker actions in our training dataset
##.value_counts() — Counts how frequently each action occurs
##.idxmax() — Returns the action code with the highest count
baseline_action = train["action"].value_counts().idxmax()
baseline_accuracy = (test["action"]==baseline_action).mean()

print("Bayesian model accuracy:", round(accuracy*100,2), "%")
print("Baseline accuracy:", round(baseline_accuracy*100, 2),"%")

##Create table of actual versus predicted actions
table = pd.crosstab(
    results["action"],
    results["prediction"],
    rownames=["Actual"],
    colnames=["Predicted"]
)

##Include all three actions, even those never predicted
table = table.reindex(
    index=[0,1,2],
    columns=[0,1,2],
    fill_value=0
)

print (table.rename(
    index=action_labels,
    columns=action_labels
))

Bayesian model accuracy: 63.42 %
Baseline accuracy: 47.8 %
Predicted    Fold  Check/call  Bet/Raise
Actual                                  
Fold        39735          10          0
Check/call  10822       12996          0
Bet/Raise   12132        7450          0


In [6]:
import os
print (os.getcwd())

c:\Users\kirst\Desktop\Dartmouth\phh-dataset
